# D2-07 Case study — Regionalised hydrogen pathways

This final Day 2 case study combines a foreground import, explicit electricity relinking, and regionalised LCIA with `edges`.

We compare three electrolysis routes—**AEC**, **PEM**, and **SOEC**—using Spanish low-voltage electricity and all ImpactWorld+ 2.1 damage-oriented methods.

> **Screening-study boundary:** the functional unit is 1 kg of hydrogen at each dataset's own output pressure: AEC at 20 bar, PEM at 30 bar, and SOEC at 1 bar. The comparison therefore includes pressure differences and is not a perfectly pressure-equivalent technology comparison.

## Learning goals

After this notebook, you should be able to:

- import and validate a bundled Brightway foreground
- relink one direct technosphere exchange to a country-specific supplier
- select a family of packaged `edges` methods
- calculate several technologies across many regionalised methods
- compare rankings without adding distinct LCIA indicators together
- interpret location-level contributions and method-coverage gaps

## Workflow

The case study follows six steps:

1. import the hydrogen foreground;
2. select the three comparable production activities;
3. relink their direct electricity demand to Spain;
4. collect ImpactWorld+ damage methods;
5. run the technology–method matrix;
6. interpret rankings and location contributions.

The import and relinking cells change the active Brightway project. They are written to be safely rerun.

## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087–3101. https://doi.org/10.1007/s11367-025-02551-7
- Bulle, C., Margni, M., Patouillard, L., Boulay, A.-M., Bourgault, G., De Bruille, V., Cao, V., Hauschild, M., Henderson, A., Humbert, S., Kashef-Haghighi, S., Kounina, A., Laurent, A., Levasseur, A., Liard, G., Rosenbaum, R. K., Roy, P.-O., Shaked, S., & Jolliet, O. (2019). IMPACT World+: a globally regionalized life cycle impact assessment method. *The International Journal of Life Cycle Assessment, 24*, 1653–1674. https://doi.org/10.1007/s11367-019-01583-0

The bundled workbook `assets/lci_hydrogen_electrolysis_v3_10_importable.xlsx` contains foreground inventories originally prepared for ecoinvent 3.10.

## 1) Import the hydrogen foreground

Importing is separated into small checks:

1. verify the project, workbook, and background databases;
2. apply the standard Excel-import strategies;
3. assign Spanish locations to hydrogen-production datasets;
4. link foreground and biosphere exchanges;
5. validate and write the database.

In [ ]:
import contextlib
import io
import json
from pathlib import Path
from textwrap import fill

import bw2data as bd
import bw2io as bi
import edges
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from edges import EdgeLCIA


PROJECT_NAME = "aalborg-rlcia-2026"
BAFU_DATABASE = "bafu"
BIOSPHERE_DATABASE = "ecoinvent-3.10-biosphere"
FOREGROUND_PATH = Path("assets/lci_hydrogen_electrolysis_v3_10_importable.xlsx")

bd.projects.set_current(PROJECT_NAME)

### 1.1 Preflight checks

Fail early with a clear message if the workbook or either required background database is missing.

In [ ]:
if not FOREGROUND_PATH.exists():
    raise FileNotFoundError(f"Foreground workbook not found: {FOREGROUND_PATH}")

required_databases = {BAFU_DATABASE, BIOSPHERE_DATABASE}
missing_databases = required_databases.difference(bd.databases)

if missing_databases:
    raise ValueError(
        "Missing required Brightway databases: "
        + ", ".join(sorted(missing_databases))
    )

pd.Series({
    "project": bd.projects.current,
    "foreground workbook": FOREGROUND_PATH.name,
    "background database": BAFU_DATABASE,
    "biosphere database": BIOSPHERE_DATABASE,
}, name="Import inputs")

### 1.2 Read and normalize the workbook

`ExcelImporter` reads the workbook, and `apply_strategies()` normalizes units, uncertainty fields, and internal links. The detailed strategy log is hidden so the output stays focused.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    importer = bi.ExcelImporter(FOREGROUND_PATH)
    importer.apply_strategies()

hydrogen_database_name = importer.db_name

print(f"Foreground database name: {hydrogen_database_name}")
print(f"Datasets read from workbook: {len(importer.data)}")

In [ ]:
hydrogen_dataset_count = 0

for dataset in importer.data:
    if "hydrogen production" not in dataset["name"].lower():
        continue

    dataset["location"] = "ES"
    hydrogen_dataset_count += 1

    for exchange in dataset["exchanges"]:
        if exchange.get("type") == "production":
            exchange["location"] = "ES"

print(f"Hydrogen-production datasets assigned to ES: {hydrogen_dataset_count}")

### 1.4 Link foreground and biosphere exchanges

The first call resolves links inside the foreground. The next two calls link technosphere exchanges to `bafu` and elementary flows to the biosphere database.

In [ ]:
technosphere_fields = ("name", "reference product", "location", "unit")

importer.match_database(fields=technosphere_fields)
importer.match_database(
    BAFU_DATABASE,
    fields=technosphere_fields,
)
importer.match_database(
    BIOSPHERE_DATABASE,
    kind="biosphere",
    fields=("name", "categories", "unit"),
)

### 1.5 Validate and write

The final two importer statistics should be zero. Otherwise, one or more exchanges remain unlinked and the foreground should not be written.

In [ ]:
import_statistics = importer.statistics()

print(
    "Importer statistics "
    "(datasets, exchanges, unique unlinked, total unlinked):",
    import_statistics,
)

if tuple(import_statistics[-2:]) != (0, 0):
    raise ValueError("The foreground still contains unlinked exchanges")

Delete an earlier copy of the foreground before writing. This makes the notebook reproducible when run more than once.

The writer may warn that not every foreground dataset has a geocollection. That does not prevent this exercise: the activities use country codes, and `edges` later applies its own geographic matching strategies.

In [ ]:
if hydrogen_database_name in bd.databases:
    del bd.databases[hydrogen_database_name]

importer.write_database()

hydrogen_database = bd.Database(hydrogen_database_name)
print(
    f"Created {hydrogen_database_name!r} "
    f"with {len(hydrogen_database):,} activities."
)

### 1.6 Verify the hydrogen activities

The workbook contains four hydrogen-production datasets. The SOEC route with an external steam input is retained in the database but excluded from the three-technology comparison.

In [ ]:
imported_hydrogen_rows = [
    {
        "name": activity["name"],
        "reference product": activity["reference product"],
        "location": activity["location"],
        "unit": activity["unit"],
    }
    for activity in hydrogen_database
    if "hydrogen production" in activity["name"].lower()
]

imported_hydrogen_table = (
    pd.DataFrame(imported_hydrogen_rows)
    .sort_values("name")
    .reset_index(drop=True)
)
imported_hydrogen_table

In [ ]:
def find_one_activity(database, *, name, reference_product, location):
    matches = [
        activity
        for activity in database
        if activity["name"] == name
        and activity["reference product"] == reference_product
        and activity.get("location") == location
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one match for {name!r} in {location}; "
            f"found {len(matches)}"
        )
    return matches[0]

Select the three foreground activities by exact metadata. Keeping these specifications visible makes the functional-unit differences explicit.

In [ ]:
hydrogen_specs = {
    "AEC": {
        "name": "hydrogen production, gaseous, 20 bar, from AEC electrolysis, from grid electricity",
        "reference_product": "hydrogen, gaseous, 20 bar",
    },
    "PEM": {
        "name": "hydrogen production, gaseous, 30 bar, from PEM electrolysis, from grid electricity",
        "reference_product": "hydrogen, gaseous, 30 bar",
    },
    "SOEC": {
        "name": "hydrogen production, gaseous, 1 bar, from SOEC electrolysis, from grid electricity",
        "reference_product": "hydrogen, gaseous, 1 bar",
    },
}

hydrogen_activities = {
    technology: find_one_activity(
        hydrogen_database,
        name=specification["name"],
        reference_product=specification["reference_product"],
        location="DK",
    )
    for technology, specification in hydrogen_specs.items()
}

In [ ]:
electricity_rows = []

for technology, activity in hydrogen_activities.items():
    for e in activity.technosphere():
        if e["unit"] == "kilowatt hour":
            print(technology, e["amount"], "kWh/kg H2")

The plot makes an important foreground difference visible before LCIA: SOEC has the lowest direct electricity demand, while PEM has the highest.

## 3) Select ImpactWorld+ 2.1 damage methods

`edges.get_available_methods()` returns method tuples. Keep only methods whose family is `ImpactWorld+ 2.1` and whose final level is `damage`.

Read each JSON file once to obtain its descriptive name and unit.

In [ ]:
edges_data_directory = Path(edges.__file__).resolve().parent / "data"
available_damage_methods = sorted(
    [
        method
        for method in edges.get_available_methods()
        if method[0] == "ImpactWorld+ 2.1"
        and method[-1] == "damage"
    ],
    key=lambda method: method[1],
)

method_rows = []
for method in available_damage_methods:
    method_path = edges_data_directory / f"{'_'.join(method)}.json"
    metadata = json.loads(method_path.read_text(encoding="utf-8"))
    method_rows.append({
        "method": method,
        "short_label": method[1],
        "name": metadata["name"],
        "unit": metadata.get("unit"),
    })

damage_method_table = pd.DataFrame(method_rows)

Damage methods fall into two endpoint-unit families:

- `DALY` for human health;
- `PDF.m2.yr` for ecosystem quality.

Methods can be compared within each unit, but they should not be added together: each row represents a distinct LCIA question.

In [ ]:
method_counts = (
    damage_method_table.groupby("unit")
    .size()
    .reindex(["DALY", "PDF.m2.yr"])
)

fig, ax = plt.subplots(figsize=(6.5, 3.5))
bars = ax.bar(
    method_counts.index,
    method_counts.values,
    color=["#457B9D", "#2A9D8F"],
)
ax.bar_label(bars, padding=3)
ax.set_title("ImpactWorld+ damage methods by endpoint unit")
ax.set_ylabel("Number of methods")
ax.set_ylim(0, method_counts.max() * 1.2)
plt.tight_layout()
plt.show()

print(f"Total damage methods selected: {len(damage_method_table)}")

## 4) Calculate the technology–method matrix

There are 33 methods and 3 technologies, giving 99 LCIA scores.

For each method:

- initialize `EdgeLCIA` for the first technology;
- try `redo_lcia()` for the next technologies;
- rebuild the LCA only if reuse is unsupported;
- aggregate characterized impacts by **consumer location**.

The helper suppresses repetitive mapping messages while retaining one concise progress line per method.

In [ ]:
def calculate_or_redo(lca, activity, method):
    with contextlib.redirect_stdout(io.StringIO()):
        if lca is not None:
            try:
                lca.redo_lcia({activity.id: 1})
                return lca
            except RuntimeError:
                pass

        lca = EdgeLCIA(demand={activity: 1}, method=method)
        lca.lci()
        lca.apply_strategies()
        lca.evaluate_cfs()
        lca.lcia()

    return lca

The second helper converts the characterized exchange table into one contribution per consumer location. It also returns the number of matched exchange rows as a coverage diagnostic.

In [ ]:
def summarize_location_contributions(lca):
    with contextlib.redirect_stdout(io.StringIO()):
        cf_table = lca.generate_cf_table(include_unmatched=False)

    if cf_table.empty or "impact" not in cf_table:
        return 0, pd.DataFrame(columns=["consumer location", "impact"])

    matched_rows = cf_table.dropna(subset=["impact"]).copy()
    location_summary = (
        matched_rows.assign(
            consumer_location=lambda frame: (
                frame["consumer location"].fillna("Unknown")
            )
        )
        .groupby("consumer_location", as_index=False)["impact"]
        .sum()
        .rename(columns={"consumer_location": "consumer location"})
    )

    return len(matched_rows), location_summary

Run the matrix. The result table stores one row per technology and method; the location table stores the additive geographic decomposition.

This is the longest calculation in the notebook.

In [ ]:
technology_order = ["AEC", "PEM", "SOEC"]
score_rows = []
location_rows = []
method_total = len(damage_method_table)

for method_index, method_row in enumerate(
    damage_method_table.itertuples(index=False),
    start=1,
):
    print(
        f"[{method_index:02d}/{method_total}] "
        f"{method_row.short_label}"
    )
    lca = None

    for technology in technology_order:
        activity = hydrogen_activities[technology]
        lca = calculate_or_redo(lca, activity, method_row.method)
        matched_count, location_summary = summarize_location_contributions(lca)

        score_rows.append({
            "technology": technology,
            "method": method_row.short_label,
            "unit": method_row.unit,
            "score": float(lca.score),
            "matched exchange rows": matched_count,
        })

        for location_row in location_summary.itertuples(index=False):
            location_rows.append({
                "technology": technology,
                "method": method_row.short_label,
                "unit": method_row.unit,
                "consumer location": location_row[0],
                "impact": float(location_row.impact),
            })

hydrogen_results = pd.DataFrame(score_rows)
hydrogen_location_results = pd.DataFrame(location_rows)

### 4.1 Validate coverage and reconciliation

Two checks make the large result matrix auditable:

- every technology–method combination should produce one score row;
- location contributions should sum back to the LCIA score.

A method with three exact zero scores has no characterized coverage for these inventories and is excluded from ranking plots.

In [ ]:
method_has_coverage = (
    hydrogen_results.groupby("method")["score"]
    .apply(lambda scores: (scores != 0).any())
)
zero_coverage_methods = method_has_coverage.index[~method_has_coverage].tolist()

location_totals = (
    hydrogen_location_results
    .groupby(["technology", "method"], as_index=False)["impact"]
    .sum()
    .rename(columns={"impact": "location total"})
)
reconciliation = hydrogen_results.merge(
    location_totals,
    on=["technology", "method"],
    how="left",
)
reconciliation["location total"] = reconciliation["location total"].fillna(0)
reconciliation["difference"] = (
    reconciliation["score"] - reconciliation["location total"]
)

print(f"Score rows: {len(hydrogen_results)}")
print(f"Methods with characterized coverage: {method_has_coverage.sum()}")
print(f"Methods without coverage: {zero_coverage_methods}")
print(
    "Maximum score–location reconciliation difference:",
    f'{reconciliation["difference"].abs().max():.2e}',
)

## 5) Compare technology rankings

For ranking stability, keep the 33 methods separate and divide each score by the PEM score for the same method:

\[
\text{relative score} =
\frac{\text{technology score}}{\text{PEM score}}
\]

- `1.00×` means equal to PEM;
- below `1.00×` means a lower score than PEM;

Section 5.4 later gives a complementary within-endpoint sum by midpoint contribution.
- above `1.00×` means a higher score.

This normalization preserves each indicator's ranking while avoiding unit and scale problems.

In [ ]:
score_matrix = (
    hydrogen_results
    .pivot(index="method", columns="technology", values="score")
    .reindex(columns=technology_order)
)
unit_by_method = (
    damage_method_table
    .set_index("short_label")["unit"]
    .reindex(score_matrix.index)
)

covered_methods = method_has_coverage[method_has_coverage].index
covered_scores = score_matrix.loc[covered_methods]
relative_to_pem = covered_scores.div(covered_scores["PEM"], axis=0)

relative_groups = {
    "Human health (DALY)": relative_to_pem.loc[unit_by_method == "DALY"],
    "Ecosystem quality (PDF.m2.yr)": relative_to_pem.loc[
        unit_by_method == "PDF.m2.yr"
    ],
}

The heatmap helper keeps one row per method and one column per technology. Lower relative scores are greener.

In [ ]:
def plot_relative_score_heatmap(relative_scores, title):
    plot_data = relative_scores.sort_values("SOEC")
    row_count = len(plot_data)
    figure_height = max(4, 0.42 * row_count)
    values = plot_data.to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(8.5, figure_height))
    image = ax.imshow(
        values,
        aspect="auto",
        cmap="RdYlGn_r",
        vmin=min(0.7, np.nanmin(values)),
        vmax=max(1.1, np.nanmax(values)),
    )

    ax.set_xticks(range(len(plot_data.columns)))
    ax.set_xticklabels(plot_data.columns)
    ax.set_yticks(range(row_count))
    ax.set_yticklabels([fill(label, 44) for label in plot_data.index], fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("Hydrogen technology")

    for row_index in range(row_count):
        for column_index in range(len(plot_data.columns)):
            value = values[row_index, column_index]
            ax.text(
                column_index,
                row_index,
                f"{value:.2f}×",
                ha="center",
                va="center",
                fontsize=8,
            )

    colorbar = fig.colorbar(image, ax=ax, pad=0.02)
    colorbar.set_label("Score relative to PEM")
    plt.tight_layout()
    plt.show()

### 5.1 Human-health methods

In [ ]:
plot_relative_score_heatmap(
    relative_groups["Human health (DALY)"],
    "Human-health damage scores relative to PEM",
)

### 5.2 Ecosystem-quality methods

In [ ]:
plot_relative_score_heatmap(
    relative_groups["Ecosystem quality (PDF.m2.yr)"],
    "Ecosystem-quality damage scores relative to PEM",
)

### 5.3 Ranking stability

Count which technology has the lowest score for each covered method. A tie is recorded separately instead of being assigned arbitrarily.

In [ ]:
winner_rows = []

for method, technology_scores in covered_scores.iterrows():
    minimum_score = technology_scores.min()
    is_winner = np.isclose(
        technology_scores,
        minimum_score,
        rtol=1e-10,
        atol=0,
    )
    winners = technology_scores.index[is_winner].tolist()

    winner_rows.append({
        "method": method,
        "unit": unit_by_method[method],
        "winner": winners[0] if len(winners) == 1 else "Tie",
    })

winner_table = pd.DataFrame(winner_rows)
winner_counts = (
    winner_table.groupby(["unit", "winner"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=["DALY", "PDF.m2.yr"])
    .reindex(columns=technology_order + ["Tie"], fill_value=0)
)

ax = winner_counts.plot(
    kind="bar",
    figsize=(7.5, 4),
    color=[technology_colors[name] for name in technology_order] + ["#999999"],
)
ax.set_title("Lowest-scoring technology by endpoint method")
ax.set_xlabel("Endpoint unit")
ax.set_ylabel("Number of methods")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Winner", frameon=False)
plt.tight_layout()
plt.show()

winner_counts

### 5.4 Endpoint totals by midpoint-indicator contribution

The ranking plots keep each method separate. A complementary question is: **which midpoint indicators build the total endpoint result for each electrolyser?**

Within each endpoint unit, sum the compatible damage contributions:

- human-health contributions form a total in `DALY`;
- ecosystem-quality contributions form a total in `PDF.m2.yr`.

To keep the legends readable, the eight largest indicators are shown separately and all remaining contributions are grouped as `Other indicators`. DALY and `PDF.m2.yr` are never combined with each other.

In [ ]:
def make_endpoint_composition(unit, top_n=8):
    unit_methods = unit_by_method[
        (unit_by_method == unit)
        & unit_by_method.index.isin(covered_scores.index)
    ].index

    # Rows become technologies; columns become midpoint indicators.
    indicator_scores = (
        covered_scores.loc[unit_methods]
        .T
        .reindex(technology_order)
    )
    indicator_importance = (
        indicator_scores.abs().sum(axis=0).sort_values(ascending=False)
    )
    top_indicators = indicator_importance.head(top_n).index
    composition = indicator_scores.loc[:, top_indicators].copy()

    remaining_indicators = indicator_scores.columns.difference(top_indicators)
    if len(remaining_indicators):
        composition["Other indicators"] = (
            indicator_scores.loc[:, remaining_indicators].sum(axis=1)
        )

    return composition

In [ ]:
endpoint_specs = [
    ("Human health", "DALY"),
    ("Ecosystem quality", "PDF.m2.yr"),
]
endpoint_compositions = {
    label: make_endpoint_composition(unit)
    for label, unit in endpoint_specs
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (label, unit) in zip(axes, endpoint_specs):
    composition = endpoint_compositions[label]
    bottom = np.zeros(len(composition))
    indicator_colors = plt.cm.tab20(
        np.linspace(0, 1, len(composition.columns))
    )

    for indicator, color in zip(composition.columns, indicator_colors):
        values = composition[indicator].to_numpy(dtype=float)
        ax.bar(
            composition.index,
            values,
            bottom=bottom,
            color=color,
            edgecolor="white",
            linewidth=0.6,
            label=fill(indicator, 28),
        )
        bottom += values

    endpoint_totals = composition.sum(axis=1)
    label_offset = endpoint_totals.max() * 0.025
    for position, total in enumerate(endpoint_totals):
        ax.text(
            position,
            total + label_offset,
            f"{total:.3g}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    ax.set_title(f"{label}: total by midpoint indicator")
    ax.set_xlabel("Electrolyser technology")
    ax.set_ylabel(unit)
    ax.set_ylim(0, endpoint_totals.max() * 1.16)
    ax.legend(
        title="Midpoint indicators",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.16),
        ncol=2,
        frameon=False,
        fontsize=8,
        title_fontsize=9,
    )

plt.tight_layout()
plt.show()

**How to read the charts:** bar height gives the summed endpoint result, while segment size shows the contribution from each midpoint indicator. Compare segment patterns as well as totals—two technologies can have similar totals for different reasons.

## 6) Interpret location contributions

The heatmaps show rankings but not *where* the characterized impacts occur. Use two representative regionalised methods:

- `Particulate matter formation` for human health;
- `Land occupation, biodiversity` for ecosystem quality.

For each method, retain the six consumer locations with the largest absolute contributions and aggregate the remainder as `Other locations`.

In [ ]:
def make_location_breakdown(method_label, top_n=6):
    method_data = hydrogen_location_results[
        hydrogen_location_results["method"] == method_label
    ]
    pivot = (
        method_data
        .pivot_table(
            index="consumer location",
            columns="technology",
            values="impact",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(columns=technology_order, fill_value=0)
    )

    location_importance = pivot.abs().sum(axis=1).sort_values(ascending=False)
    top_locations = location_importance.head(top_n).index
    breakdown = pivot.loc[top_locations].copy()

    if len(pivot) > len(top_locations):
        breakdown.loc["Other locations"] = pivot.drop(index=top_locations).sum()

    return breakdown

In [ ]:
representative_methods = [
    ("Particulate matter formation", "DALY"),
    ("Land occupation, biodiversity", "PDF.m2.yr"),
]
location_breakdowns = {
    method: make_location_breakdown(method)
    for method, _ in representative_methods
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, (method, unit) in zip(axes, representative_methods):
    breakdown = location_breakdowns[method].iloc[::-1]
    breakdown.plot(
        kind="barh",
        ax=ax,
        color=[technology_colors[name] for name in technology_order],
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(fill(method, 34))
    ax.set_xlabel(unit)
    ax.set_ylabel("Consumer location")
    ax.legend(title="Technology", frameon=False)

plt.tight_layout()
plt.show()

for method, _ in representative_methods:
    dominant_location = (
        location_breakdowns[method]
        .drop(index="Other locations", errors="ignore")
        .abs()
        .sum(axis=1)
        .idxmax()
    )
    print(f"{method}: largest displayed location contribution = {dominant_location}")

## Guided exercise: inspect another indicator

Change `chosen_method` to any covered method shown in the heatmaps. The plot ranks the three technologies using the method's original unit.

Try `Water availability, human health` or `Freshwater ecotoxicity, long term`.

In [ ]:
chosen_method = "Water availability, human health"

if chosen_method not in covered_scores.index:
    raise ValueError(f"{chosen_method!r} has no characterized score")

chosen_scores = covered_scores.loc[chosen_method].sort_values()
chosen_unit = unit_by_method[chosen_method]

fig, ax = plt.subplots(figsize=(6.5, 3.6))
bars = ax.barh(
    chosen_scores.index,
    chosen_scores.values,
    color=[technology_colors[name] for name in chosen_scores.index],
)
ax.bar_label(bars, fmt="%.3g", padding=3)
ax.set_title(fill(chosen_method, 46))
ax.set_xlabel(chosen_unit)
ax.set_ylabel("Technology")
plt.tight_layout()
plt.show()

### Interpretation questions

- Does the same technology have the lowest score across all covered methods?
- How closely does the ranking follow direct electricity demand?
- Which methods show the largest difference between AEC and PEM?
- Which consumer locations dominate particulate matter and land occupation?
- What additional inventory work would be required to compare all routes at the same delivery pressure?

## Key cautions

- The three hydrogen products have different output pressures.
- Only direct electrolyser electricity is relinked to Spain.
- The relative-score heatmaps compare LCIA methods separately.
- The stacked endpoint charts sum damage contributions only within their compatible endpoint unit; never combine `DALY` with `PDF.m2.yr`.
- An exact zero score can indicate missing method coverage, not zero environmental relevance.
- Location plots aggregate by the **consumer location recorded on characterized edges**; this is not automatically the same as an emission's physical location.

## Recap

You have now:

- imported and validated a hydrogen foreground;
- relinked direct AEC, PEM, and SOEC electricity demand to Spain;
- selected 33 ImpactWorld+ damage methods;
- calculated and reconciled 99 regionalised LCIA scores;
- compared rankings using unitless scores relative to PEM;
- decomposed total DALY and ecosystem-quality results by midpoint indicator;
- counted ranking wins and interpreted representative location contributions.